In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as functional
from torch import nn
from torchvision import datasets, transforms


# ============================================================
# CONFIGURATION
# ============================================================


DATASET_ROOT = Path("data")
FLOAT_MODEL_PATH = Path("models/cifar10_alexnet10.pth")
QUANTIZED_MODEL_PATH = Path("models/quantized/cifar_alexnet_int8.npz")
QUANTIZATION_CONFIG_PATH = Path("models/quantized/quantization_config.json")

OUTPUT_DIRECTORY = Path("integer_results")

TEST_IMAGE_INDEX = 1

REQUANT_SHIFT = 24

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]


class CifarAlexNet(nn.Module):
    def __init__(self, number_of_classes: int = 10) -> None:
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 8, 5, stride=1, padding=2),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(8, 16, 3, stride=1, padding=1),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, 3, stride=1, padding=1),
            nn.ReLU(inplace=False),

            nn.Conv2d(32, 32, 3, stride=1, padding=1),
            nn.ReLU(inplace=False),

            nn.Conv2d(32, 16, 3, stride=1, padding=1),
            nn.ReLU(inplace=False),

            nn.MaxPool2d(2, 2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(inplace=False),

            nn.Linear(64, 32),
            nn.ReLU(inplace=False),

            nn.Linear(32, number_of_classes),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        features = self.features(inputs)
        flattened = torch.flatten(features, start_dim=1)

        return self.classifier(flattened)


def quantize_tensor(values: torch.Tensor, scale: float) -> torch.Tensor:
    if scale <= 0:
        raise ValueError(f"Scale must be positive, received {scale}")

    quantized = torch.round(values.to(torch.float64) / scale)
    quantized = torch.clamp(quantized, -128, 127)

    return quantized.to(torch.int8)


def float_multiplier_to_q_int(floating_multiplier: float, shift: int) -> int:
    if floating_multiplier < 0:
        raise ValueError("Requantization multiplier must be non-negative")

    integer_multiplier = int(round(floating_multiplier * (1 << shift)))

    if integer_multiplier > (2**31 - 1):
        raise OverflowError(
            f"Multiplier {integer_multiplier} does not fit in signed 32 bits"
        )

    return integer_multiplier


def requantize_accumulator(
    accumulator: torch.Tensor,
    integer_multiplier: int,
    shift: int,
    apply_relu: bool,
) -> torch.Tensor:
    acc = accumulator.to(torch.int64)
    mult = int(integer_multiplier)

    product = acc * mult

    sign = torch.sign(product)
    absolute_product = torch.abs(product)

    if shift == 0:
        rounded_magnitude = absolute_product
    else:
        divisor = 1 << shift
        half = 1 << (shift - 1)

        quotient = absolute_product // divisor
        remainder = absolute_product % divisor

        round_up = (remainder > half) | (
            (remainder == half) & (quotient % 2 == 1)
        )

        rounded_magnitude = torch.where(round_up, quotient + 1, quotient)

    scaled = sign * rounded_magnitude

    if apply_relu:
        scaled = torch.clamp(scaled, min=0)

    scaled = torch.clamp(scaled, -128, 127)

    return scaled.to(torch.int8)


def integer_conv2d(
    inputs: torch.Tensor,
    weights: np.ndarray,
    biases: np.ndarray,
    integer_multiplier: int,
    shift: int,
    kernel_size: int,
    stride: int,
    padding: int,
    apply_relu: bool,
) -> torch.Tensor:
    if inputs.dtype != torch.int8:
        raise TypeError(f"Expected INT8 convolution input, got {inputs.dtype}")

    batch_size = inputs.shape[0]
    output_channels = weights.shape[0]

    unfolded = functional.unfold(
        inputs.to(torch.float64),
        kernel_size=kernel_size,
        padding=padding,
        stride=stride,
    )

    weight_matrix = torch.from_numpy(
        weights.astype(np.int64)
    ).to(torch.float64).reshape(output_channels, -1)

    accumulators = torch.matmul(weight_matrix.unsqueeze(0), unfolded)

    bias_tensor = torch.from_numpy(
        biases.astype(np.int64)
    ).to(torch.float64).reshape(1, output_channels, 1)

    accumulators = accumulators + bias_tensor

    input_height = inputs.shape[2]
    input_width = inputs.shape[3]

    output_height = (input_height + (2 * padding) - kernel_size) // stride + 1
    output_width = (input_width + (2 * padding) - kernel_size) // stride + 1

    accumulators = accumulators.reshape(
        batch_size, output_channels, output_height, output_width
    ).to(torch.int64)

    return requantize_accumulator(accumulators, integer_multiplier, shift, apply_relu)


def integer_max_pool2d(
    inputs: torch.Tensor,
    kernel_size: int = 2,
    stride: int = 2,
) -> torch.Tensor:
    if inputs.dtype != torch.int8:
        raise TypeError(f"Expected INT8 pooling input, got {inputs.dtype}")

    pooled = functional.max_pool2d(
        inputs.to(torch.float64), kernel_size=kernel_size, stride=stride
    )

    return pooled.to(torch.int8)


def integer_linear(
    inputs: torch.Tensor,
    weights: np.ndarray,
    biases: np.ndarray,
    integer_multiplier: int,
    shift: int,
    apply_relu: bool,
) -> torch.Tensor:
    if inputs.dtype != torch.int8:
        raise TypeError(f"Expected INT8 linear input, got {inputs.dtype}")

    input_int64 = inputs.to(torch.int64)
    weight_int64 = torch.from_numpy(weights.astype(np.int64))
    bias_int64 = torch.from_numpy(biases.astype(np.int64))

    accumulators = torch.matmul(input_int64, weight_int64.transpose(0, 1))
    accumulators = accumulators + bias_int64.reshape(1, -1)

    return requantize_accumulator(accumulators, integer_multiplier, shift, apply_relu)


def print_tensor_summary(name: str, tensor: torch.Tensor) -> None:
    minimum = int(tensor.min().item())
    maximum = int(tensor.max().item())

    print(
        f"{name:8s} "
        f"shape={str(tuple(tensor.shape)):18s} "
        f"min={minimum:4d} "
        f"max={maximum:4d}"
    )


def save_tensor(name: str, tensor: torch.Tensor) -> None:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

    np.save(OUTPUT_DIRECTORY / f"{name}.npy", tensor.cpu().numpy())


def load_test_image(image_index: int) -> tuple[torch.Tensor, int, int]:
    transform = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ]
    )

    dataset = datasets.CIFAR10(
        root=DATASET_ROOT, train=False, download=True, transform=transform
    )

    normalized_image, original_label = dataset[image_index]

    true_class = original_label

    return normalized_image.unsqueeze(0), original_label, true_class


def load_float_model() -> CifarAlexNet:
    checkpoint = torch.load(FLOAT_MODEL_PATH, map_location="cpu", weights_only=False)

    model = CifarAlexNet(number_of_classes=10)

    state_dict = checkpoint.get("model_state_dict", checkpoint)

    model.load_state_dict(state_dict)
    model.eval()

    return model


def main() -> None:
    if not QUANTIZED_MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Quantized model not found: {QUANTIZED_MODEL_PATH}\n"
            "Run quantize_model.py first."
        )

    if not QUANTIZATION_CONFIG_PATH.exists():
        raise FileNotFoundError(
            f"Quantization configuration not found: {QUANTIZATION_CONFIG_PATH}"
        )

    quantized_model = np.load(QUANTIZED_MODEL_PATH)

    with QUANTIZATION_CONFIG_PATH.open("r", encoding="utf-8") as file:
        configuration = json.load(file)

    activation_scales = configuration["activation_scales"]

    layer_names = ["conv1", "conv2", "conv3", "conv4", "conv5", "fc6", "fc7", "fc8"]

    integer_multipliers = {}

    for layer_name in layer_names:
        float_multiplier = float(quantized_model[f"{layer_name}_multiplier"])
        integer_multipliers[layer_name] = float_multiplier_to_q_int(
            float_multiplier, REQUANT_SHIFT
        )

    print()
    print("========================================")
    print("Q-format integer multipliers (must match")
    print("cifar_requantization.vh)")
    print("========================================")

    for layer_name in layer_names:
        print(f"{layer_name:5s} | Q{REQUANT_SHIFT}={integer_multipliers[layer_name]}")

    image, original_label, true_class = load_test_image(TEST_IMAGE_INDEX)

    # --------------------------------------------------------
    # Floating-point reference
    # --------------------------------------------------------

    float_model = load_float_model()

    with torch.no_grad():
        float_logits = float_model(image)

    float_prediction = int(float_logits.argmax(dim=1).item())

    # --------------------------------------------------------
    # Quantize input
    # --------------------------------------------------------

    input_scale = float(configuration["input_scale"])

    x = quantize_tensor(values=image, scale=input_scale)

    print()
    print("========================================")
    print("Integer layer outputs (bit-exact w/ RTL)")
    print("========================================")

    print_tensor_summary("Input", x)
    save_tensor("input", x)

    # --------------------------------------------------------
    # Conv1 and Pool1
    # --------------------------------------------------------

    x = integer_conv2d(
        inputs=x,
        weights=quantized_model["conv1_weight"],
        biases=quantized_model["conv1_bias"],
        integer_multiplier=integer_multipliers["conv1"],
        shift=REQUANT_SHIFT,
        kernel_size=5, stride=1, padding=2,
        apply_relu=True,
    )
    print_tensor_summary("Conv1", x)
    save_tensor("conv1", x)

    x = integer_max_pool2d(x)
    print_tensor_summary("Pool1", x)
    save_tensor("pool1", x)

    # --------------------------------------------------------
    # Conv2 and Pool2
    # --------------------------------------------------------

    x = integer_conv2d(
        inputs=x,
        weights=quantized_model["conv2_weight"],
        biases=quantized_model["conv2_bias"],
        integer_multiplier=integer_multipliers["conv2"],
        shift=REQUANT_SHIFT,
        kernel_size=3, stride=1, padding=1,
        apply_relu=True,
    )
    print_tensor_summary("Conv2", x)
    save_tensor("conv2", x)

    x = integer_max_pool2d(x)
    print_tensor_summary("Pool2", x)
    save_tensor("pool2", x)

    # --------------------------------------------------------
    # Conv3
    # --------------------------------------------------------

    x = integer_conv2d(
        inputs=x,
        weights=quantized_model["conv3_weight"],
        biases=quantized_model["conv3_bias"],
        integer_multiplier=integer_multipliers["conv3"],
        shift=REQUANT_SHIFT,
        kernel_size=3, stride=1, padding=1,
        apply_relu=True,
    )
    print_tensor_summary("Conv3", x)
    save_tensor("conv3", x)

    # --------------------------------------------------------
    # Conv4
    # --------------------------------------------------------

    x = integer_conv2d(
        inputs=x,
        weights=quantized_model["conv4_weight"],
        biases=quantized_model["conv4_bias"],
        integer_multiplier=integer_multipliers["conv4"],
        shift=REQUANT_SHIFT,
        kernel_size=3, stride=1, padding=1,
        apply_relu=True,
    )
    print_tensor_summary("Conv4", x)
    save_tensor("conv4", x)

    # --------------------------------------------------------
    # Conv5 and Pool5
    # --------------------------------------------------------

    x = integer_conv2d(
        inputs=x,
        weights=quantized_model["conv5_weight"],
        biases=quantized_model["conv5_bias"],
        integer_multiplier=integer_multipliers["conv5"],
        shift=REQUANT_SHIFT,
        kernel_size=3, stride=1, padding=1,
        apply_relu=True,
    )
    print_tensor_summary("Conv5", x)
    save_tensor("conv5", x)

    x = integer_max_pool2d(x)
    print_tensor_summary("Pool5", x)
    save_tensor("pool5", x)

    x = torch.flatten(x, start_dim=1)
    print_tensor_summary("Flatten", x)
    save_tensor("flatten", x)

    # --------------------------------------------------------
    # FC6
    # --------------------------------------------------------

    x = integer_linear(
        inputs=x,
        weights=quantized_model["fc6_weight"],
        biases=quantized_model["fc6_bias"],
        integer_multiplier=integer_multipliers["fc6"],
        shift=REQUANT_SHIFT,
        apply_relu=True,
    )
    print_tensor_summary("FC6", x)
    save_tensor("fc6", x)

    # --------------------------------------------------------
    # FC7
    # --------------------------------------------------------

    x = integer_linear(
        inputs=x,
        weights=quantized_model["fc7_weight"],
        biases=quantized_model["fc7_bias"],
        integer_multiplier=integer_multipliers["fc7"],
        shift=REQUANT_SHIFT,
        apply_relu=True,
    )
    print_tensor_summary("FC7", x)
    save_tensor("fc7", x)

    # --------------------------------------------------------
    # FC8
    # --------------------------------------------------------

    integer_logits = integer_linear(
        inputs=x,
        weights=quantized_model["fc8_weight"],
        biases=quantized_model["fc8_bias"],
        integer_multiplier=integer_multipliers["fc8"],
        shift=REQUANT_SHIFT,
        apply_relu=False,
    )
    print_tensor_summary("FC8", integer_logits)
    save_tensor("fc8", integer_logits)

    integer_prediction = int(integer_logits.argmax(dim=1).item())

    fc8_scale = float(activation_scales["fc8"])
    dequantized_logits = integer_logits.to(torch.float64) * fc8_scale

    print()
    print("========================================")
    print("Integer inference result")
    print("========================================")
    print(f"Dataset index: {TEST_IMAGE_INDEX}")
    print(f"Original CIFAR label: {original_label}")
    print(f"True class: {true_class} - {CLASS_NAMES[true_class]}")
    print()
    print(f"Float logits:       {float_logits.numpy().flatten()}")
    print(f"INT8 logits:        {integer_logits.numpy().flatten()}")
    print(f"Dequantized logits: {dequantized_logits.numpy().flatten()}")
    print()
    print(f"Float prediction: {float_prediction} - {CLASS_NAMES[float_prediction]}")
    print(f"INT8 prediction:  {integer_prediction} - {CLASS_NAMES[integer_prediction]}")

    if integer_prediction == float_prediction:
        print("Prediction agreement: PASS")
    else:
        print("Prediction agreement: FAIL")

    if integer_prediction == true_class:
        print("INT8 classification: PASS")
    else:
        print("INT8 classification: INCORRECT")

    print("========================================")


if __name__ == "__main__":
    main()